# Human benchmark — cleaned annotations and AI validation
Run top to bottom in Colab. No paid calls, model training, or rescoring. The package contains cleaned CSVs, the original coding schema, an audit and analysis helper. Originals were not modified.

**Important:** the supplied files contain 49 annotated News rows in A and 50 in B, yielding **49 shared news items**; both also contain X annotations. Default `SCOPE='News'` follows your requested headline analysis. Set `SCOPE='Both'` only if the X annotations are intended for inclusion; shared X count is determined by IDs, not assumed.

Cleaning repaired reversible character-encoding corruption and normalized whitespace, case and unambiguous aliases (`Price-levels`, `National aggregate`, `Future`). Noncanonical relevance values (`Other`, `Irrelevant`) were retained, not guessed into detailed categories. Text entered in `emotional_quote` was moved to `emotional_quote_excerpt` and preserved in `emotional_quote_original`; the categorical status is left blank/unassessed. Free-text zero placeholders were removed with an audit trail. No missing labels were filled from AI or other answers.

This remains below the assignment's 300-item requirement. The subset may not retain the original sample's stratification; report its actual source/class distributions and small denominators.
One shared X text had a single extra leading @ in B; comparison text was normalized to the original A input and B’s prior text retained. X and combined analyses now pass text checks.


In [ ]:
from pathlib import Path
import sys,json,zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Upload the COMPLETE human_benchmark_package.zip if the package is not already extracted.
candidates=[Path('/content/human_benchmark_package'),Path.cwd(),Path.cwd()/'human_benchmark_package']
PACKAGE=next((p for p in candidates if (p/'benchmark_tools.py').exists()),None)
if PACKAGE is None:
    from google.colab import files
    uploaded=files.upload()
    archives=[name for name in uploaded if name.endswith('.zip')]
    if len(archives)!=1:raise ValueError('Upload the single supplied ZIP package.')
    PACKAGE=Path('/content/human_benchmark_package');PACKAGE.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        for name in archive.namelist():
            if not (PACKAGE/name).resolve().is_relative_to(PACKAGE.resolve()):raise ValueError('Unsafe archive path')
        archive.extractall(PACKAGE)
sys.path.insert(0,str(PACKAGE))
import benchmark_tools as bt
SCOPE='News'  # News, X, or Both. Keep News for the requested headline benchmark.
assert SCOPE in ['News','X','Both']
# Preserve decisions on rerun; each scope gets separate outputs.
OUT=PACKAGE/('results_v2_'+SCOPE.lower());OUT.mkdir(exist_ok=True)
RULES=bt.schema_at(PACKAGE)
A,B=bt.load_pair(PACKAGE,SCOPE)
print('Shared annotated IDs:',len(A))
print(A.platform.value_counts())
display(bt.read_csv(PACKAGE/'input_summary.csv'))
print('Cleaned inputs:',PACKAGE)

## 1. Data-quality review and agreement before adjudication
Coverage is field-specific: neither blank agreement nor arbitrary recoding is counted. Detailed relevance agreement only uses canonical labels. An auxiliary coarse relevance result collapses explicit exclusion categories and B's generic `Irrelevant` into `Excluded`; A's generic `Other` remains unresolved. It does not replace the detailed relevance result.

Percentage-agreement intervals use Wilson's binomial approximation. They do not account for stratified sampling or dependence. Cohen's kappa is unweighted; undefined kappa from constant labels stays missing. Sparse-source kappa is descriptive.

In [ ]:
audit=bt.read_csv(PACKAGE/'values_needing_review.csv')
audit=audit.loc[audit.item_id.isin(A.item_id)]
display(audit)
audit.to_csv(OUT/'input_values_needing_review.csv',index=False)
agreement,distributions=bt.agreement_tables(A,B,RULES)
agreement.to_csv(OUT/'human_agreement.csv',index=False)
distributions.to_csv(OUT/'human_label_distributions.csv',index=False)
display(agreement.loc[agreement.subset.eq('all_shared')])

# Preserve exclusion-rule contradictions for discussion, not automatic correction.
logical=[]
for who,frame in [('A',A),('B',B)]:
    for _,row in frame.iterrows():
        if row.political_attribution in ['Neither','Not applicable'] and (row.attribution_voice not in ['Not applicable',''] or row.attribution_target not in ['','0','0.0']):
            logical.append(dict(annotator=who,item_id=row.item_id,issue='Attribution voice/target inconsistent with Neither/Not applicable'))
        if bt.coarse_relevance(row.relevance)=='Excluded':
            binary=[f for f in RULES if f.startswith('topic_') or f in ['layoffs','hiring','rising_prices','falling_inflation','growth_expansion','recession_risk']]
            if any(row[f]=='1' for f in binary):logical.append(dict(annotator=who,item_id=row.item_id,issue='Excluded text retains positive topic/direction flags'))
pd.DataFrame(logical,columns=['annotator','item_id','issue']).to_csv(OUT/'logical_review_flags.csv',index=False)
print('Logical review flags:',len(logical))
core=['relevance','economic_tone','emotional_wording','primary_topic','price_frame','experience_frame','time_frame','political_attribution']
plot=agreement.loc[agreement.subset.eq('all_shared')&agreement.field.isin(core)].set_index('field').reindex(core)
fig,ax=plt.subplots(figsize=(10,4));ax.barh(plot.index,plot.agreement*100,color='#426A8C');ax.set_xlim(0,100);ax.set_xlabel('Agreement among valid paired answers (%)');ax.invert_yaxis()
for i,(_,row) in enumerate(plot.iterrows()):
    if pd.notna(row.agreement):ax.text(min(row.agreement*100+1,94),i,f'n={int(row.n)}',va='center',fontsize=9)
fig.tight_layout();fig.savefig(OUT/'human_agreement.png',dpi=180);plt.show()

## 2. Adjudication without looking at AI predictions
`adjudication_worklist.csv` contains one row per disputed, missing or noncanonical field. Original answers remain intact. Fill `final_label`, `reason` and `adjudicator` only after reviewing the guide and discussing the text. Do not simply copy AI labels. If neither person labeled a category (for example emotional-quote status), report this as incomplete or explicitly document the later joint annotation; it is not an original independent rating.

Agreed valid labels are prefilled into a provisional benchmark. Until disagreements are resolved, it is an agreement-selected subset and can make apparent accuracy optimistic. Final metrics always state field-specific gold coverage. Avoid claiming an adjudicated benchmark is complete while substantial fields remain unresolved.

In [ ]:
WORK=OUT/'adjudication_worklist.csv'
work=bt.make_adjudication(A,B,RULES,WORK)
print('Fields awaiting decisions:',work.final_label.eq('').sum())
print('Edit this file, then rerun the NEXT cell:',WORK)
display(work[['item_id','field','A_original','B_original','issue','final_label']].head(20))
# Optional Colab download: uncomment, edit the CSV, and re-upload/replace it at WORK.
# from google.colab import files
# files.download(str(WORK))

In [ ]:
# Rerun after editing adjudication_worklist.csv. No original annotations are overwritten.
work=bt.read_csv(WORK)
gold=bt.build_gold(A,B,RULES,work)
gold.to_csv(OUT/'consensus_and_adjudicated_labels.csv',index=False)
coverage=pd.DataFrame([dict(field=f,shared_items=len(gold),gold_labels=gold[f].isin(labels).sum()) for f,labels in RULES.items()])
coverage['unresolved']=coverage.shared_items-coverage.gold_labels
coverage.to_csv(OUT/'gold_coverage_by_field.csv',index=False)
display(coverage)
print('Completed manual decisions:',work.final_label.ne('').sum())
print('Rows fully resolved across all coded fields:',gold[list(RULES)].ne('').all(axis=1).sum())

## 3. Connect saved AI outputs and the original split/source key
No API calls. Mount Drive and read Module 03 outputs. Optional paths can be changed if your project moved. If the original manifest is missing, accuracy is descriptive only: no new split is invented. All development items remain development. Do not tune the classifier on original held-out labels before reporting held-out accuracy.

Agreement-by-source and AI-by-source results can be sparse; report counts, not rankings of outlets. Missing/failed AI outputs are counted as `Missing` errors, not silently dropped.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass
ROOT=Path('/content/drive/MyDrive/NLP-FinalProject')
STUDY=ROOT/'03_ai_measurement/headline-measurement-v1'
AI_RUN=STUDY/'model_runs/4ff1cce4591eb134'
AI_PATH=AI_RUN/'ai_predictions_all_statuses.csv'
MANIFEST_PATH=STUDY/'human_annotation/human_sample_manifest.csv'
metadata=bt.read_csv(MANIFEST_PATH) if MANIFEST_PATH.exists() else pd.DataFrame({'item_id':gold.item_id,'source':'Unknown','split':'Unknown'})
assert not metadata.item_id.duplicated().any()
print('Original source/split file:',MANIFEST_PATH,'| exists:',MANIFEST_PATH.exists())
key=gold[['item_id','platform']].merge(metadata[[c for c in ['item_id','source','split','analysis_month'] if c in metadata]],on='item_id',how='left',validate='one_to_one')
if 'split' in key:display(pd.crosstab(key.platform,key.split.fillna('Unknown')))
key.to_csv(OUT/'evaluation_source_split_key.csv',index=False)
# Field-specific source agreement using only original independent annotations.
source_rows=[]
if 'source' in metadata:
    for source,group in metadata.loc[metadata.item_id.isin(A.item_id)].groupby('source'):
        ids=set(group.item_id);aa=A.loc[A.item_id.isin(ids)].reset_index(drop=True);bb=B.set_index('item_id').loc[aa.item_id].reset_index()
        table,_=bt.agreement_tables(aa,bb,RULES);table['source']=source;source_rows.append(table)
if source_rows:
    pd.concat(source_rows,ignore_index=True).to_csv(OUT/'human_agreement_by_source.csv',index=False)
print('Saved AI file:',AI_PATH,'| exists:',AI_PATH.exists())

## 4. AI class/source accuracy and matched dictionary baseline
This cell runs when the saved AI CSV exists. Core relevance/geography/genre are evaluated on available field-level gold labels. Substantive fields additionally require gold relevance = Relevant. An erroneous AI exclusion remains an error. Full AI tone metrics retain Mixed/Unclear/Not applicable if present in gold. Dictionary comparison uses exactly the same gold-Relevant Positive/Negative/Neutral items for both methods. Dictionary vocabulary matches Module 03 (`transparent-tone-v1`).

The `all_completed_descriptive` scope is not a new held-out test. Only `original_heldout` uses the preserved original held-out split. If adjudication is unfinished, results are provisional consensus-subset results and must be labeled as such. No confidence in population-wide performance follows from these small, partially labeled samples.

In [ ]:
if AI_PATH.exists():
    ai=bt.read_csv(AI_PATH)
    # Check that accepted predictions refer to the same annotated text where text exists.
    if 'text' in ai:
        matched=gold[['item_id','text']].merge(ai[['item_id','text']],on='item_id',suffixes=('_human','_ai'),validate='one_to_one')
        normalized=lambda s:' '.join(str(s).split())
        assert matched.text_human.map(normalized).eq(matched.text_ai.map(normalized)).all(),'Human/AI texts differ; inspect IDs and versions.'
    metrics,summary,comparison,evaluated=bt.evaluate(gold,ai,metadata,RULES)
    metrics.to_csv(OUT/'ai_class_metrics.csv',index=False)
    summary.to_csv(OUT/'ai_summary_by_field_source.csv',index=False)
    comparison.to_csv(OUT/'ai_vs_dictionary.csv',index=False)
    evaluated.to_csv(OUT/'evaluation_rows.csv',index=False)
    print('Field-specific AI accuracy (not all rows are adjudicated):')
    if len(summary):display(summary.loc[summary.grouping.eq('overall')])
    print('AI and dictionary on identical three-class gold-tone rows:');display(comparison)
    pending=work.final_label.eq('').sum()
    if pending:print(f'PROVISIONAL: {pending} disputed/missing field decisions remain. Agreement-only selection may inflate accuracy.')
    for scope in ['all_completed_descriptive','original_heldout']:
        sub=evaluated.loc[evaluated.relevance_gold.eq('Relevant')&evaluated.economic_tone_gold.isin(RULES['economic_tone'])].copy()
        if scope=='original_heldout':sub=sub.loc[sub.split.eq('heldout')]
        if sub.empty:continue
        predicted=sub.economic_tone_ai.fillna('Missing')
        predicted=predicted.where(predicted.isin(RULES['economic_tone']),'Missing')
        matrix=pd.crosstab(sub.economic_tone_gold,predicted)
        matrix.to_csv(OUT/f'tone_confusion_{scope}.csv')
        fig,ax=plt.subplots(figsize=(7,5));image=ax.imshow(matrix.to_numpy(),cmap='Blues',aspect='auto')
        ax.set_xticks(range(len(matrix.columns)));ax.set_xticklabels(matrix.columns,rotation=35,ha='right')
        ax.set_yticks(range(len(matrix.index)));ax.set_yticklabels(matrix.index)
        for i in range(len(matrix)):
            for j in range(len(matrix.columns)):ax.text(j,i,str(matrix.iloc[i,j]),ha='center',va='center')
        ax.set_xlabel('AI');ax.set_ylabel('Human consensus/adjudicated');ax.set_title(f'Tone: {scope}; n={len(sub)}')
        fig.colorbar(image,ax=ax);fig.tight_layout();fig.savefig(OUT/f'tone_confusion_{scope}.png',dpi=180);plt.show()
else:
    print('AI file unavailable. Agreement/adjudication outputs are complete; set AI_PATH and rerun this cell for accuracy.')

## 5. Interpretation and optional dashboard integration
Report shared items, paired-valid counts per field, raw agreement, kappa and class distributions. Do not treat raw excerpts in `emotional_quote` as independent categorical ratings. Report adjudication procedures and unresolved decisions. Report field-specific class/source accuracy and original held-out results only when sufficient labels exist. Human accuracy evaluation does not establish factual calibration or reader cognitive bias.

The supplied forms contain more than headlines, but the default analysis uses only the 49 shared News IDs; it does not invent A's missing 50th answer. If you include X later, change SCOPE and rerun, preserving the original split. Do not infer that a completed first subset of a stratified template remains representative.

Outputs stay inside this package. For the existing dashboard, copy the appropriate completed `human_agreement.csv` and held-out overall rows of `ai_summary_by_field_source.csv` to the dashboard's data folder after checking actual sample counts. Label validation panels with the actual shared sample, replacing older 70-item notes. Do not overwrite the original Module 03 annotation files.

In [ ]:
print('FINAL OUTPUT FOLDER:',OUT)
for path in sorted(OUT.glob('*')):print(path.name)
# Optional download of analysis outputs only:
# import shutil
# from google.colab import files
# files.download(shutil.make_archive(str(OUT), 'zip', OUT))